In [2]:
import subprocess, os

# Set up kaggle API key first
os.makedirs("/root/.config/kaggle", exist_ok=True)
with open("/root/.config/kaggle/kaggle.json", "w") as f:
    f.write('{"username":"bajointerns","key":"fe6395b567bf06c54788889310683cb2"}')
os.chmod("/root/.config/kaggle/kaggle.json", 0o600)

# Try downloading DiverseVul
result = subprocess.run(
    ["kaggle", "datasets", "download", "-d", "vccolombo/diversevul", "--unzip"],
    capture_output=True, text=True
)
print(result.stdout)
print(result.stderr)
print(os.listdir("."))

403 Client Error: Forbidden for url: https://api.kaggle.com/v1/datasets.DatasetApiService/GetDatasetMetadata


['.virtual_documents']


In [3]:
import subprocess, os

os.makedirs("/root/.config/kaggle", exist_ok=True)
with open("/root/.config/kaggle/kaggle.json", "w") as f:
    f.write('{"username":"bajointerns","key":"fe6395b567bf06c54788889310683cb2"}')
os.chmod("/root/.config/kaggle/kaggle.json", 0o600)

# Search for diversevul
result = subprocess.run(
    ["kaggle", "datasets", "list", "--search", "diversevul"],
    capture_output=True, text=True
)
print(result.stdout)
print(result.stderr)

ref                                                                 title                                                   size  lastUpdated                 downloadCount  voteCount  usabilityRating  
------------------------------------------------------------------  ------------------------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
furduisorinoctavian/vulnerable-c-functions-for-security-prediction  Vulnerable C Functions for Security Prediction 🔐    56470517  2025-03-15 12:54:15.553000            147          4                1  
rasabakhtiari/diversevul-bof-binary                                 diversevul bof binary                                  33604  2025-10-30 14:57:56.857000              0          0        0.3529412  
snehaakp/diversevul                                                 DiverseVul                                         121033016  2026-02-27 05:11:16.837000              2          0          

In [4]:
!kaggle datasets download -d ahmedtabib/diversevul-cleaned --unzip
import os
print(os.listdir("."))

Dataset URL: https://www.kaggle.com/datasets/ahmedtabib/diversevul-cleaned
License(s): unknown
100%|████████████████████████████████████████| 110M/110M [00:04<00:00, 26.1MB/s]

['.virtual_documents', 'DiverseVul-Cleaned.csv']


In [5]:
import pandas as pd

df = pd.read_csv("DiverseVul-Cleaned.csv")
print("Total rows:", len(df))
print("Columns:", list(df.columns))
print("\nLabel distribution:")
print(df.iloc[:, -1].value_counts())  # check last column for labels
print("\nFirst row:")
print(df.head(1))

Total rows: 319549
Columns: ['Unnamed: 0', 'func', 'target', 'cwe', 'project', 'commit_id', 'hash', 'size', 'message']

Label distribution:
message
CImg<>::load_bmp() and CImg<>::load_pandore(): Check that dimensions encoded in file does not exceed file size.                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     

/tmp/ipykernel_57/378945457.py:3: DtypeWarning: Columns (2,7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("DiverseVul-Cleaned.csv")


In [6]:
print("Label distribution:")
print(df['target'].value_counts())
print("\nTotal:", len(df))
print("Missing code:", df['func'].isna().sum())

Label distribution:
target
0                         170598
0                         131046
1                          17879
['CWE-190', 'CWE-125']        11
['CWE-617']                    1
['CWE-703', 'CWE-770']         1
Name: count, dtype: int64

Total: 319549
Missing code: 0


In [7]:
import pandas as pd
import re

df = pd.read_csv("DiverseVul-Cleaned.csv", low_memory=False)

# Clean target column — keep only rows where target is clearly 0 or 1
def clean_target(val):
    try:
        v = int(float(str(val).strip()))
        if v in [0, 1]:
            return v
    except:
        pass
    return None

df['label'] = df['target'].apply(clean_target)
df = df.dropna(subset=['label'])
df['label'] = df['label'].astype(int)

print("After cleaning:")
print(df['label'].value_counts())
print("Total usable rows:", len(df))

After cleaning:
label
0    301644
1     17879
Name: count, dtype: int64
Total usable rows: 319523


In [8]:
import re

def clean_code(code):
    code = re.sub(r'//.*', '', code)
    code = re.sub(r'/\*.*?\*/', '', code, flags=re.DOTALL)
    code = re.sub(r'\n\s*\n', '\n', code)
    return code.strip()

df['clean_code'] = df['func'].apply(clean_code)

# Balance: all vulnerable + 3x safe
vuln = df[df['label'] == 1]
safe = df[df['label'] == 0].sample(n=len(vuln)*3, random_state=42)
df_balanced = pd.concat([vuln, safe]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Vulnerable: {len(df_balanced[df_balanced['label']==1])}")
print(f"Safe:       {len(df_balanced[df_balanced['label']==0])}")
print(f"Total:      {len(df_balanced)}")

# Save for later
df_balanced[['clean_code', 'label']].to_csv('diversevul_balanced.csv', index=False)
print("Saved to diversevul_balanced.csv!")

Vulnerable: 17879
Safe:       53637
Total:      71516
Saved to diversevul_balanced.csv!
